
# Stunting Risk Model - Training (Local / Antigravity)

**Goal:** Generate `stunting_model.json` for the NutriGuard / SeribuAsa AI early-warning service.

This notebook trains a lightweight, explainable Logistic Regression model for **3-month stunting risk** using the same inputs available in the application:

- child age and sex;
- latest weight, height, and optional MUAC;
- WHO-style weight and height z-scores;
- recent growth trend from previous measurement;
- latest FIES food insecurity score.

**Important scope note:** the cohort is synthetic because no real Posyandu/Puskesmas dataset is included in this repository. The generator is calibrated for Indonesian demo/product behavior, not for clinical deployment. Before production health decisions, replace or recalibrate this notebook with real local measurement data.

**Steps:**
1. Check/install dependencies in the active notebook environment.
2. Generate a calibrated Indonesian synthetic cohort.
3. Train Logistic Regression with app-aligned features.
4. Evaluate AUC, calibration, and app thresholds (`low`, `medium`, `high`).
5. Export training artifacts to `apps/backend/ml/stunting/artifacts/` and deploy a runtime copy to `apps/backend/app/services/stunting_model.json`.
6. Restart the backend so `stunting_risk_service.py` loads the exported model.

**Runtime:** about 1 minute on a normal laptop CPU. No GPU needed.



## 0. Environment check


In [ ]:

import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'apps' / 'backend' / 'app' / 'services').exists():
            return candidate
    raise RuntimeError(
        'Could not find project root containing apps/backend/app/services. '
        f'Started from: {start}'
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SERVICE_DIR = PROJECT_ROOT / 'apps' / 'backend' / 'app' / 'services'
ARTIFACT_DIR = PROJECT_ROOT / 'apps' / 'backend' / 'ml' / 'stunting' / 'artifacts'

print(f'Python: {sys.version}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Runtime model directory: {SERVICE_DIR}')
print(f'Training artifact directory: {ARTIFACT_DIR}')



## 1. Install dependencies

Run this only if the active notebook kernel does not already have these packages. The backend runtime does **not** need these ML dependencies; they are only for training/exporting the JSON artifact.


In [ ]:

%pip install numpy==2.1.3 scikit-learn==1.6.1 pandas==2.2.2


## 2. Configuration

**IMPORTANT:** `FEATURE_NAMES` order MUST match `StuntingFeatures.as_model_input()` in `apps/backend/app/services/stunting_risk_service.py`. Do not reorder.

In [ ]:

from __future__ import annotations
import json
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

# Must match StuntingFeatures.as_model_input() in apps/backend/app/services/stunting_risk_service.py.
# Do not reorder this list unless you also update the runtime service.
FEATURE_NAMES: List[str] = [
    'age_months', 'is_male', 'weight_kg', 'height_cm', 'muac_cm',
    'z_score_weight', 'z_score_height',
    'delta_z_height', 'delta_z_weight',
    'days_since_last', 'trend_score', 'fies_score',
]

# Raw weight/height are already represented by z-scores and age/sex.
# Keeping them as model inputs often creates confusing coefficients because height is highly age-dependent.
# They stay in FEATURE_NAMES for runtime compatibility, but exported coefficients are set to 0.
MODEL_FEATURE_NAMES: List[str] = [
    'age_months', 'is_male', 'muac_cm',
    'z_score_weight', 'z_score_height',
    'delta_z_height', 'delta_z_weight',
    'days_since_last', 'trend_score', 'fies_score',
]
EXCLUDED_MODEL_FEATURES = sorted(set(FEATURE_NAMES) - set(MODEL_FEATURE_NAMES))

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_MODEL_FILE = SERVICE_DIR / 'stunting_model.json'
MODEL_FILE = ARTIFACT_DIR / 'stunting_model.json'
METRICS_FILE = ARTIFACT_DIR / 'stunting_model_metrics.json'

# Runtime uses these same buckets.
THRESHOLD_MEDIUM = 0.35
THRESHOLD_HIGH = 0.65
MODEL_VERSION = 'logreg-v3-id-synthetic-calibrated'

N_SAMPLES = 15_000
SEED = 42

# Configurable target for Indonesian demo calibration.
# Update this value if your project decides to follow a newer official prevalence number.
TARGET_STUNTING_PREVALENCE = 0.215
TARGET_CURRENT_SEVERE_SHARE = 0.055



## 3. Synthetic data generator

The generator below is intentionally **not pure random noise**. It creates a coherent synthetic Indonesian balita cohort:

- a latent household vulnerability factor affects FIES, follow-up gaps, MUAC, and growth velocity;
- age/sex determine approximate WHO median weight and height;
- z-scores drive measured weight/height so the model inputs stay consistent with the app;
- the label is derived from simulated **future HAZ at 3 months**, not directly from the same rule used by the model;
- the final future-stunting label is calibrated to a configurable Indonesian target prevalence.

This is still synthetic. Treat it as a reproducible prototype until real local measurement data are available.


In [ ]:

@dataclass
class CohortConfig:
    n_samples: int = N_SAMPLES
    seed: int = SEED
    target_stunting_prevalence: float = TARGET_STUNTING_PREVALENCE
    target_current_severe_share: float = TARGET_CURRENT_SEVERE_SHARE


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def _who_weight_median(age_months, is_male):
    """Approximate WHO weight-for-age median (kg), enough for synthetic generation."""
    base = np.where(
        age_months < 6,
        3.3 + age_months * 0.7,
        np.where(
            age_months < 24,
            7.5 + (age_months - 6) * 0.22,
            11.5 + (age_months - 24) * 0.18,
        ),
    )
    return base + is_male * 0.3


def _who_height_median(age_months, is_male):
    """Approximate WHO height-for-age median (cm), enough for synthetic generation."""
    base = np.where(
        age_months < 6,
        50 + age_months * 2.8,
        np.where(
            age_months < 24,
            66 + (age_months - 6) * 1.1,
            85 + (age_months - 24) * 0.7,
        ),
    )
    return base + is_male * 0.6


def _calibrate_future_haz_to_target(raw_future_haz: np.ndarray, target_rate: float) -> np.ndarray:
    """Shift future HAZ so P(future HAZ < -2) matches the target prevalence."""
    target_rate = float(np.clip(target_rate, 0.05, 0.60))
    cutoff_quantile = np.quantile(raw_future_haz, target_rate)
    shift = -2.0 - cutoff_quantile
    return raw_future_haz + shift


def _calibrate_current_haz_severe_share(current_haz: np.ndarray, target_share: float) -> np.ndarray:
    """Shift current HAZ so severe examples (HAZ < -3) are present in training."""
    target_share = float(np.clip(target_share, 0.01, 0.20))
    cutoff_quantile = np.quantile(current_haz, target_share)
    shift = -3.0 - cutoff_quantile
    return current_haz + shift


def generate_cohort(cfg: CohortConfig) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.seed)
    n = cfg.n_samples

    # App population: children 0-60 months. Triangular distribution gives more toddler records,
    # which is common in routine monitoring compared with a perfectly uniform age mix.
    age_months = np.clip(np.rint(rng.triangular(0, 24, 60, size=n)), 0, 59).astype(int)
    is_male = rng.binomial(1, 0.51, size=n)

    # Synthetic Indonesian context. These variables are NOT model inputs; they only shape plausible data.
    # 1 = urban/peri-urban, 0 = rural/remote. Regional disadvantage is a coarse latent stratum.
    is_urban = rng.binomial(1, 0.48, size=n)
    regional_disadvantage = rng.choice([0.0, 0.35, 0.75], size=n, p=[0.55, 0.30, 0.15])
    household_shock = rng.gamma(shape=1.3, scale=0.35, size=n)
    vulnerability = (
        rng.normal(0, 1, size=n)
        + 0.45 * regional_disadvantage
        + 0.35 * (1 - is_urban)
        + 0.30 * household_shock
    )

    # FIES score, 0..8. Higher vulnerability means higher food insecurity,
    # but noise keeps households with similar vulnerability from being identical.
    fies_raw = 1.8 + 1.15 * vulnerability + rng.normal(0, 1.15, size=n)
    fies_score = np.clip(np.rint(fies_raw), 0, 8).astype(int)

    weight_median = _who_weight_median(age_months, is_male)
    height_median = _who_height_median(age_months, is_male)

    # Current anthropometric status. HAZ has a heavier left tail so the model sees
    # normal, moderate, and severe examples. WAZ is correlated with HAZ but not identical.
    tail_prob = 0.06 + 0.18 * sigmoid_np(vulnerability - 0.7)
    tail_shortfall = rng.binomial(1, tail_prob, size=n) * rng.gamma(shape=1.4, scale=0.55, size=n)
    z_score_height = rng.normal(-0.35 - 0.48 * vulnerability, 0.70, size=n) - tail_shortfall
    z_score_height = _calibrate_current_haz_severe_share(
        z_score_height,
        cfg.target_current_severe_share,
    )
    z_score_height = np.clip(z_score_height, -4.5, 3.0)

    z_score_weight = 0.55 * z_score_height + rng.normal(-0.10 * vulnerability, 0.62, size=n)
    z_score_weight = np.clip(z_score_weight, -4.2, 3.5)

    # MUAC in cm. Optional in the app, but when present it should align with weight/growth status.
    muac_cm = 14.2 + 0.42 * z_score_weight + 0.16 * z_score_height - 0.18 * vulnerability + rng.normal(0, 0.45, size=n)
    muac_cm = np.clip(muac_cm, 9.0, 18.0)

    # Convert z-scores back to raw measurements so app-visible fields remain internally consistent.
    # SDs are approximate synthetic values; production diagnosis should still rely on official WHO tables.
    height_sd = 2.4 + 0.018 * age_months
    weight_sd = 0.75 + 0.018 * age_months
    height_cm = np.maximum(45.0, height_median + z_score_height * height_sd)
    weight_kg = np.maximum(2.0, weight_median + z_score_weight * weight_sd)

    # Recent trend from previous visit. Negative trend is more common under vulnerability/high FIES.
    delta_z_height = rng.normal(0.04 - 0.08 * vulnerability - 0.025 * fies_score, 0.28, size=n)
    delta_z_weight = rng.normal(0.03 - 0.06 * vulnerability - 0.020 * fies_score, 0.25, size=n)
    delta_z_height = np.clip(delta_z_height, -1.4, 1.1)
    delta_z_weight = np.clip(delta_z_weight, -1.2, 1.1)

    days_since_last = np.clip(
        rng.normal(45 + 12 * fies_score + 14 * (1 - is_urban) + 8 * regional_disadvantage, 18, size=n),
        14,
        240,
    ).astype(int)

    trend_score = np.where(
        delta_z_height > 0.3,
        1.0,
        np.where(delta_z_height < -0.3, -1.0, 0.0),
    )

    # True synthetic target: projected HAZ in 3 months. This creates a real horizon label,
    # instead of labeling from the same score the model will learn.
    raw_future_haz = (
        z_score_height
        + 0.75 * delta_z_height
        - 0.030 * fies_score
        - 0.070 * np.maximum(0, 12.5 - muac_cm)
        - 0.030 * np.maximum(0, (days_since_last - 90) / 30.0)
        + rng.normal(0, 0.32, size=n)
    )
    future_z_score_height = _calibrate_future_haz_to_target(
        raw_future_haz,
        cfg.target_stunting_prevalence,
    )
    label = (future_z_score_height < -2.0).astype(int)

    df = pd.DataFrame({
        'age_months': age_months,
        'is_male': is_male,
        'weight_kg': weight_kg.round(2),
        'height_cm': height_cm.round(1),
        'muac_cm': muac_cm.round(2),
        'z_score_weight': z_score_weight.round(3),
        'z_score_height': z_score_height.round(3),
        'delta_z_height': delta_z_height.round(3),
        'delta_z_weight': delta_z_weight.round(3),
        'days_since_last': days_since_last,
        'trend_score': trend_score,
        'fies_score': fies_score.astype(float),
        'future_z_score_height': future_z_score_height.round(3),
        'label': label,
    })

    quality = {
        'future_stunting_rate': float(df['label'].mean()),
        'current_stunting_rate_haz_lt_minus_2': float((df['z_score_height'] < -2).mean()),
        'current_severe_rate_haz_lt_minus_3': float((df['z_score_height'] < -3).mean()),
        'mean_fies': float(df['fies_score'].mean()),
    }
    print('[data] synthesized', n, 'samples')
    print('[data] future stunting target/actual:', f'{cfg.target_stunting_prevalence:.1%}', '/', f'{quality["future_stunting_rate"]:.1%}')
    print('[data] current HAZ<-2:', f'{quality["current_stunting_rate_haz_lt_minus_2"]:.1%}', 'HAZ<-3 target/actual:', f'{cfg.target_current_severe_share:.1%}', '/', f'{quality["current_severe_rate_haz_lt_minus_3"]:.1%}')
    print('[data] mean FIES:', f'{quality["mean_fies"]:.2f}')

    return df


def data_quality_report(df: pd.DataFrame) -> Dict:
    """Quick checks to catch random-looking or impossible synthetic data."""
    report = {
        'n_samples': int(len(df)),
        'future_stunting_rate': float(df['label'].mean()),
        'current_stunting_rate_haz_lt_minus_2': float((df['z_score_height'] < -2).mean()),
        'current_severe_rate_haz_lt_minus_3': float((df['z_score_height'] < -3).mean()),
        'feature_ranges': df[FEATURE_NAMES].agg(['min', 'mean', 'max']).round(4).to_dict(),
        'label_by_future_haz_band': pd.crosstab(
            pd.cut(df['future_z_score_height'], [-99, -3, -2, -1.5, -1, 99]),
            df['label'],
            normalize='index',
        ).round(4).to_dict(),
        'label_rate_by_fies': df.groupby('fies_score')['label'].mean().round(4).to_dict(),
        'corr_with_label': df[MODEL_FEATURE_NAMES + ['label']]
            .corr(numeric_only=True)['label']
            .drop('label')
            .sort_values(key=lambda s: s.abs(), ascending=False)
            .round(4)
            .to_dict(),
    }
    return report


cfg = CohortConfig()
df = generate_cohort(cfg)
quality_report = data_quality_report(df)
print('\n[quality] Correlation with label:')
print(pd.Series(quality_report['corr_with_label']).to_string())
df.head()



## 4. Train + evaluate

The model trains on `MODEL_FEATURE_NAMES`, then exports a full runtime-compatible coefficient vector. `weight_kg` and `height_cm` are intentionally assigned zero coefficients because the app already converts them into age/sex-aware z-scores.

Evaluation includes:

- ROC AUC for ranking quality;
- Brier score for calibration;
- binary precision/recall/F1 at `0.35` (medium-or-high intervention threshold);
- binary precision/recall/F1 at `0.65` (high-risk threshold);
- distribution of predicted low/medium/high buckets used by the app.


In [ ]:

def risk_bucket(scores: np.ndarray) -> np.ndarray:
    return np.where(scores >= THRESHOLD_HIGH, 'high', np.where(scores >= THRESHOLD_MEDIUM, 'medium', 'low'))


def binary_metrics_at_threshold(y_true, y_score, threshold: float) -> Dict[str, float]:
    y_pred = (y_score >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'predicted_positive_rate': float(y_pred.mean()),
    }


def train(df: pd.DataFrame, seed: int) -> Tuple[LogisticRegression, StandardScaler, Dict]:
    idx = np.arange(len(df))
    y_all = df['label'].to_numpy(dtype=int)

    train_idx, test_idx = train_test_split(
        idx, test_size=0.2, stratify=y_all, random_state=seed
    )
    train_idx, val_idx = train_test_split(
        train_idx, test_size=0.15, stratify=y_all[train_idx], random_state=seed
    )

    X_train = df.iloc[train_idx][MODEL_FEATURE_NAMES].to_numpy(dtype=float)
    X_val = df.iloc[val_idx][MODEL_FEATURE_NAMES].to_numpy(dtype=float)
    X_test = df.iloc[test_idx][MODEL_FEATURE_NAMES].to_numpy(dtype=float)
    y_train = y_all[train_idx]
    y_val = y_all[val_idx]
    y_test = y_all[test_idx]

    print(f'[split] train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}')
    print(f'[split] y train/val/test = {y_train.mean():.1%} / {y_val.mean():.1%} / {y_test.mean():.1%}')

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    grid = GridSearchCV(
        LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear'),
        param_grid={'C': [0.05, 0.1, 0.5, 1.0, 2.0, 5.0], 'penalty': ['l2']},
        scoring='roc_auc', cv=5, n_jobs=-1,
    )
    grid.fit(X_train_s, y_train)
    model: LogisticRegression = grid.best_estimator_
    print(f'[tune ] best C={grid.best_params_["C"]} cv_auc={grid.best_score_:.4f}')

    val_pred = model.predict_proba(X_val_s)[:, 1]
    test_pred = model.predict_proba(X_test_s)[:, 1]
    test_label_05 = (test_pred >= 0.5).astype(int)
    test_buckets = risk_bucket(test_pred)

    metrics = {
        'val_auc': float(roc_auc_score(y_val, val_pred)),
        'test_auc': float(roc_auc_score(y_test, test_pred)),
        'test_brier_score': float(brier_score_loss(y_test, test_pred)),
        'test_accuracy_at_0_50': float(accuracy_score(y_test, test_label_05)),
        'test_precision_at_0_50': float(precision_score(y_test, test_label_05, zero_division=0)),
        'test_recall_at_0_50': float(recall_score(y_test, test_label_05, zero_division=0)),
        'test_f1_at_0_50': float(f1_score(y_test, test_label_05, zero_division=0)),
        'threshold_medium_or_high': binary_metrics_at_threshold(y_test, test_pred, THRESHOLD_MEDIUM),
        'threshold_high': binary_metrics_at_threshold(y_test, test_pred, THRESHOLD_HIGH),
        'predicted_bucket_distribution': pd.Series(test_buckets).value_counts(normalize=True).round(4).to_dict(),
        'confusion_matrix_at_0_50': confusion_matrix(y_test, test_label_05).tolist(),
        'classification_report_at_0_50': classification_report(
            y_test, test_label_05, target_names=['not_future_stunted', 'future_stunted'], output_dict=True
        ),
        'best_C': grid.best_params_['C'],
        'cv_best_auc': float(grid.best_score_),
        'model_feature_names': MODEL_FEATURE_NAMES,
        'excluded_model_features': EXCLUDED_MODEL_FEATURES,
        'test_label_rate': float(y_test.mean()),
    }
    print(f'[eval ] AUC={metrics["test_auc"]:.3f} Brier={metrics["test_brier_score"]:.3f} '
          f'F1@0.50={metrics["test_f1_at_0_50"]:.3f}')
    print('[eval ] @medium threshold:', metrics['threshold_medium_or_high'])
    print('[eval ] @high threshold:', metrics['threshold_high'])
    print('[eval ] bucket distribution:', metrics['predicted_bucket_distribution'])
    return model, scaler, metrics


model, scaler, metrics = train(df, seed=SEED)


## 5. Inspect coefficients

In [ ]:

coef_df = pd.DataFrame({
    'feature': MODEL_FEATURE_NAMES,
    'coef_scaled': model.coef_.flatten(),
})
coef_df['abs'] = coef_df['coef_scaled'].abs()
coef_df.sort_values('abs', ascending=False).reset_index(drop=True)



## 6. Export model JSON

The export is compatible with `stunting_risk_service.py`:

- `feature_names` stays equal to runtime `FEATURE_NAMES`;
- coefficients for excluded raw features are `0.0`;
- scaler parameters are included for every runtime feature;
- data quality and evaluation metrics are saved in `stunting_model_metrics.json`.


In [ ]:

def make_json_safe(obj):
    """Recursively convert pandas/numpy objects and non-string keys into JSON-safe values."""
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [make_json_safe(v) for v in obj.tolist()]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Interval):
        return str(obj)
    if pd.isna(obj) if not isinstance(obj, (dict, list, tuple, np.ndarray)) else False:
        return None
    return obj


def export_model(model, scaler, metrics, cfg, quality_report):
    trained_coef_by_name = dict(zip(MODEL_FEATURE_NAMES, model.coef_.flatten().tolist()))
    trained_mean_by_name = dict(zip(MODEL_FEATURE_NAMES, scaler.mean_.tolist()))
    trained_std_by_name = dict(zip(MODEL_FEATURE_NAMES, scaler.scale_.tolist()))

    coef = [float(trained_coef_by_name.get(name, 0.0)) for name in FEATURE_NAMES]
    means = [float(trained_mean_by_name.get(name, 0.0)) for name in FEATURE_NAMES]
    stds = [float(trained_std_by_name.get(name, 1.0)) for name in FEATURE_NAMES]
    intercept = float(model.intercept_[0])

    feature_importance = sorted(
        [{'name': n, 'coef_scaled': float(c), 'abs': abs(float(c))}
         for n, c in zip(FEATURE_NAMES, coef)],
        key=lambda x: x['abs'], reverse=True,
    )

    payload = {
        'model_version': MODEL_VERSION,
        'trained_at': datetime.now(timezone.utc).isoformat(),
        'horizon_months': 3,
        'feature_names': FEATURE_NAMES,
        'model_feature_names': MODEL_FEATURE_NAMES,
        'excluded_model_features': EXCLUDED_MODEL_FEATURES,
        'coefficients_scaled': coef,
        'intercept_scaled': intercept,
        'feature_means': means,
        'feature_stds': stds,
        'thresholds': {'medium': THRESHOLD_MEDIUM, 'high': THRESHOLD_HIGH},
        'training': {
            'n_samples': cfg.n_samples,
            'seed': cfg.seed,
            'target_stunting_prevalence': cfg.target_stunting_prevalence,
            'actual_future_stunting_rate': quality_report['future_stunting_rate'],
            'best_C': metrics['best_C'],
            'cv_best_auc': metrics['cv_best_auc'],
        },
        'feature_importance': feature_importance,
    }

    metrics_payload = {
        'model_version': MODEL_VERSION,
        'trained_at': payload['trained_at'],
        'data_quality': quality_report,
        'evaluation': metrics,
        'notes': {
            'dataset': 'synthetic calibrated Indonesian demo cohort; replace with real local data before clinical deployment',
            'label': 'future_z_score_height < -2 after simulated 3-month growth projection',
            'runtime_features': FEATURE_NAMES,
            'trained_features': MODEL_FEATURE_NAMES,
            'excluded_features_reason': 'raw weight_kg and height_cm are retained for runtime compatibility but excluded from training because z-scores already encode age/sex-adjusted anthropometry',
        },
    }

    payload = make_json_safe(payload)
    metrics_payload = make_json_safe(metrics_payload)

    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    model_json = json.dumps(payload, indent=2)
    MODEL_FILE.write_text(model_json, encoding='utf-8')
    RUNTIME_MODEL_FILE.write_text(model_json, encoding='utf-8')
    METRICS_FILE.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')
    print(f'[save ] artifact model: {MODEL_FILE}')
    print(f'[save ] runtime model:  {RUNTIME_MODEL_FILE}')
    print(f'[save ] metrics:        {METRICS_FILE}')
    return payload, metrics_payload


model_payload, metrics_payload = export_model(model, scaler, metrics, cfg, quality_report)



## 7. Export location

The previous cell writes the training artifacts to a tidy ML folder and also deploys the runtime model where the backend loader expects it:

- training copy: `apps/backend/ml/stunting/artifacts/stunting_model.json`
- metrics/audit: `apps/backend/ml/stunting/artifacts/stunting_model_metrics.json`
- runtime copy: `apps/backend/app/services/stunting_model.json`

After running the notebook, restart the backend. `stunting_risk_service.py` auto-loads the runtime `stunting_model.json` on import.


In [ ]:

print(f'Artifact model JSON: {MODEL_FILE}')
print(f'Runtime model JSON:  {RUNTIME_MODEL_FILE}')
print(f'Metrics JSON:        {METRICS_FILE}')
print('Next step after running training: restart backend and verify ACTIVE_MODEL_VERSION is the exported model_version.')



## 8. Sanity check (optional)

Predict on hand-crafted profiles using the **exported JSON payload**, not the sklearn object. This mirrors the pure-python runtime behavior in `stunting_risk_service.py`.


In [ ]:

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def predict_one_from_export(features_dict, payload):
    names = payload['feature_names']
    coefs = dict(zip(names, payload['coefficients_scaled']))
    means = dict(zip(names, payload['feature_means']))
    stds = dict(zip(names, payload['feature_stds']))
    logit = float(payload['intercept_scaled'])
    for name in names:
        raw = float(features_dict[name])
        std = float(stds[name]) or 1.0
        scaled = (raw - float(means[name])) / std
        logit += float(coefs[name]) * scaled
    score = sigmoid(logit)
    if score >= payload['thresholds']['high']:
        level = 'high'
    elif score >= payload['thresholds']['medium']:
        level = 'medium'
    else:
        level = 'low'
    return score, level


cases = {
    'high-risk': dict(age_months=24, is_male=1, weight_kg=10.5, height_cm=80, muac_cm=12.4,
                      z_score_weight=-1.6, z_score_height=-2.3, delta_z_height=-0.45,
                      delta_z_weight=-0.25, days_since_last=110, trend_score=-1.0, fies_score=6),
    'medium-risk': dict(age_months=18, is_male=0, weight_kg=9.5, height_cm=78, muac_cm=13.4,
                        z_score_weight=-0.9, z_score_height=-1.45, delta_z_height=-0.18,
                        delta_z_weight=-0.05, days_since_last=65, trend_score=0.0, fies_score=3),
    'low-risk': dict(age_months=12, is_male=0, weight_kg=10, height_cm=76, muac_cm=14.5,
                     z_score_weight=0.2, z_score_height=0.1, delta_z_height=0.25,
                     delta_z_weight=0.2, days_since_last=30, trend_score=0.0, fies_score=0),
    'healthy': dict(age_months=15, is_male=0, weight_kg=10.5, height_cm=78, muac_cm=15.0,
                    z_score_weight=0.0, z_score_height=0.0, delta_z_height=0.0,
                    delta_z_weight=0.0, days_since_last=30, trend_score=0.0, fies_score=1),
}
for name, f in cases.items():
    s, lvl = predict_one_from_export(f, model_payload)
    print(f'{name:12} score={s:.4f} level={lvl}')
